# Qwen3.5-9B — LoRA Fine-Tuning a Full Precision para Verilog

**Plataforma:** Kaggle (2× T4, 30 GB VRAM, gratis)  
**Modelo base:** `Qwen/Qwen3.5-9B`  
**Técnica:** LoRA en float16 (sin cuantización — full precision)  
**Especialidad:** AXI, UART, SPI, I2C + creación/modificación de archivos `.v`/`.sv`  
**Export:** GGUF Q8_0 para LM Studio

> **Por qué full precision es posible con 9B:**  
> 9B × 2 bytes (float16) = ~18 GB de pesos + ~4 GB activaciones/LoRA = ~22 GB total.  
> 2× T4 = 30 GB → cabe sin cuantización.  
> Sin cuantización = gradientes más precisos = mejor calidad de fine-tuning.

### Antes de empezar
1. `Settings → Accelerator → GPU T4 x2`
2. Activar **Internet** en Settings
3. Correr las celdas en orden

In [ ]:
# ── Celda 1: Verificar GPUs ───────────────────────────────────────────────────
import torch, shutil, subprocess

# nvidia-smi puede no estar en PATH hasta que se instalen drivers
smi = shutil.which('nvidia-smi') or '/usr/bin/nvidia-smi'
try:
    result = subprocess.run([smi, '--query-gpu=index,name,memory.total',
                             '--format=csv,noheader'],
                            capture_output=True, text=True, check=True)
    print('GPUs disponibles:')
    print(result.stdout)
except Exception:
    print('nvidia-smi no encontrado — verificando con torch...')

n_gpus = torch.cuda.device_count()
if n_gpus == 0:
    print('❌ No se detectaron GPUs.')
    print('   Ve a Settings → Accelerator → GPU T4 x2 y reinicia el kernel.')
else:
    total_vram = sum(
        torch.cuda.get_device_properties(i).total_memory for i in range(n_gpus)
    ) / 1e9
    for i in range(n_gpus):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')
    print(f'\nTotal GPUs : {n_gpus}')
    print(f'VRAM total : {total_vram:.1f} GB')
    if total_vram < 25:
        print('⚠️  Menos de 25 GB — asegúrate de tener T4 x2 en Settings → Accelerator')

In [ ]:
# ── Celda 2: Instalar dependencias ────────────────────────────────────────────
# ⚠️  IMPORTANTE: después de correr esta celda, haz Kernel → Restart
#                 y luego corre desde la Celda 1 otra vez.
import subprocess

subprocess.run(['pip', 'install', '-q',
    'torch==2.6.0',
    'torchvision==0.21.0',
    '--index-url', 'https://download.pytorch.org/whl/cu124',
])

subprocess.run(['pip', 'install', '-q',
    'trl',
    'datasets',
    'transformers',
    'accelerate',
    'peft',
    'tqdm',
    'gitpython',
    'huggingface_hub',
])

subprocess.run(['apt-get', 'install', '-qq', 'iverilog'])

print('✅ Dependencias instaladas.')
print('⚠️  Ahora haz: Kernel → Restart, y vuelve a correr desde la Celda 1.')

In [ ]:
# ── Celda 3: Clonar datos de entrenamiento ────────────────────────────────────
import subprocess
from pathlib import Path

DATA = Path('/kaggle/working/data/raw')

REPOS = [
    ('verilog-axi',      'https://github.com/alexforencich/verilog-axi.git'),
    ('verilog-uart',     'https://github.com/alexforencich/verilog-uart.git'),
    ('verilog-i2c',      'https://github.com/alexforencich/verilog-i2c.git'),
    ('verilog-ethernet', 'https://github.com/alexforencich/verilog-ethernet.git'),
    ('wb2axip',          'https://github.com/ZipCPU/wb2axip.git'),
    ('spi-master',       'https://github.com/nandland/spi-master.git'),
    ('zipcpu',           'https://github.com/ZipCPU/zipcpu.git'),
    ('basejump-stl',     'https://github.com/bespoke-silicon-group/basejump_stl.git'),
    ('verilog-eval',     'https://github.com/NVlabs/verilog-eval.git'),
    ('RTLLM',            'https://github.com/hkust-zhiyao/RTLLM.git'),
]

for name, url in REPOS:
    dest = DATA / name
    if dest.exists():
        print(f'  skip  {name}')
    else:
        dest.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--depth=1', url, str(dest)], capture_output=True)
        print(f'  clone {name}')

from datasets import load_dataset
hf_dest = DATA / 'hf' / 'shailja__Verilog_GitHub'
if not hf_dest.exists():
    print('  Descargando shailja/Verilog_GitHub...')
    ds = load_dataset('shailja/Verilog_GitHub')
    hf_dest.mkdir(parents=True, exist_ok=True)
    ds.save_to_disk(str(hf_dest))
    print(f'  Guardado: {len(ds["train"])} ejemplos')
else:
    print('  skip  shailja/Verilog_GitHub')

print('\nDatos listos.')

In [ ]:
# ── Celda 4: Preparar dataset ─────────────────────────────────────────────────
import json, re, random
from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_from_disk

DATA = Path('/kaggle/working/data/raw')
OUT  = Path('/kaggle/working/data/processed')
OUT.mkdir(parents=True, exist_ok=True)

SYSTEM = """You are an expert RTL designer specialised in synthesisable Verilog and SystemVerilog.
When asked to CREATE a file, output ONLY the complete file content starting with the module declaration.
When asked to MODIFY code, output the complete modified file.
Always use non-blocking assignments (<=) in clocked always blocks.
Filenames follow snake_case (e.g. axi_lite_slave.v)."""

TAG_MAP = {
    'uart':'UART serial communication', 'spi':'SPI interface',
    'i2c':'I2C interface', 'axi':'AXI bus interface',
    'axil':'AXI-Lite bus interface', 'fifo':'synchronous FIFO',
    'arb':'round-robin arbiter', 'eth':'Ethernet MAC/PHY interface',
    'wb':'Wishbone bus interface', 'fsm':'finite state machine',
}

def module_name(src):
    m = re.search(r'\bmodule\s+(\w+)', src)
    return m.group(1) if m else None

def strip_header(src):
    lines, out, in_h = src.splitlines(), [], True
    for line in lines:
        s = line.strip()
        if in_h and (s.startswith('//') or s.startswith('/*') or s == ''):
            continue
        in_h = False
        out.append(line)
    return '\n'.join(out)

def desc(path, mod):
    stem = path.stem.lower()
    for k, v in TAG_MAP.items():
        if k in stem or k in mod.lower(): return v
    return stem.replace('_', ' ')

def make_create(path, src):
    mod = module_name(src)
    if not mod: return None
    return {'messages': [
        {'role': 'system',    'content': SYSTEM},
        {'role': 'user',      'content': f'Create a file named `{path.name}` implementing a synthesisable {desc(path, mod)} module called `{mod}` in Verilog.'},
        {'role': 'assistant', 'content': src.strip()},
    ], 'task': 'create', 'file': path.name}

def make_modify(path, src):
    if 'posedge clk' not in src: return None
    mod = module_name(src)
    if not mod: return None
    modified = src.replace('posedge rst', 'negedge rst_n').replace('if (rst)', 'if (!rst_n)')
    if modified == src: return None
    return {'messages': [
        {'role': 'system',    'content': SYSTEM},
        {'role': 'user',      'content': f'File `{path.name}`: change reset to active-low (negedge rst_n). Output the complete modified file.\n\n```verilog\n{src.strip()}\n```'},
        {'role': 'assistant', 'content': modified.strip()},
    ], 'task': 'modify', 'file': path.name}

samples = []

# Repos clonados
repo_dirs = [d for d in DATA.rglob('*') if d.is_dir() and d.name not in ('verilog-eval', 'RTLLM', 'hf')]
verilog_files = [f for d in repo_dirs for f in d.rglob('*.v')] + \
                [f for d in repo_dirs for f in d.rglob('*.sv')]
print(f'Archivos Verilog: {len(verilog_files)}')

for f in tqdm(verilog_files, desc='Repos'):
    try:
        src = f.read_text(errors='ignore')
    except Exception:
        continue
    if len(src) < 100: continue
    src = strip_header(src)
    for fn in (make_create, make_modify):
        s = fn(f, src)
        if s: samples.append(s)

# HuggingFace
hf_dest = DATA / 'hf' / 'shailja__Verilog_GitHub'
if hf_dest.exists():
    ds = load_from_disk(str(hf_dest))
    split = ds['train'] if 'train' in ds else ds
    print(f'HF: {len(split)} filas raw')
    for row in tqdm(split, desc='HF dataset'):
        inst = row.get('instruction') or row.get('prompt') or ''
        out  = row.get('output') or row.get('completion') or ''
        if not inst or not out: continue
        if not any(k in out for k in ('module ', 'endmodule')): continue
        samples.append({'messages': [
            {'role': 'system',    'content': SYSTEM},
            {'role': 'user',      'content': inst.strip()},
            {'role': 'assistant', 'content': out.strip()},
        ], 'task': 'hf_instruct'})

print(f'\nTotal samples: {len(samples)}')

random.seed(42)
random.shuffle(samples)
n_valid = max(100, int(len(samples) * 0.05))
n_test  = max(100, int(len(samples) * 0.05))
splits  = {
    'test':  samples[:n_test],
    'valid': samples[n_test:n_test + n_valid],
    'train': samples[n_test + n_valid:],
}

for name, data in splits.items():
    p = OUT / f'{name}.jsonl'
    with p.open('w') as f:
        for row in data:
            f.write(json.dumps(row) + '\n')
    print(f'  {name}.jsonl: {len(data)} samples')

In [ ]:
# ── Celda 5: LoRA Fine-Tuning — Qwen3.5-9B full precision (float16) ──────────
# Sin cuantización: 9B × 2 bytes = ~18 GB + ~4 GB LoRA/activaciones = ~22 GB
# 2×T4 = 30 GB → cabe. Gradientes más precisos que con 4-bit.

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import torch, json, os
from pathlib import Path
from collections import Counter

WORK     = Path('/kaggle/working')
MODEL_ID = 'Qwen/Qwen3.5-9B'
MAX_SEQ  = 2048

n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

# ── Device map manual ─────────────────────────────────────────────────────────
cfg      = AutoConfig.from_pretrained(MODEL_ID)
n_layers = cfg.num_hidden_layers
half     = n_layers // 2
print(f"Modelo: {MODEL_ID}  |  Capas: {n_layers}")
print(f"GPU 0: capas 0–{half-1}  |  GPU 1: capas {half}–{n_layers-1}")

device_map = {"model.embed_tokens": 0}
for i in range(n_layers):
    device_map[f"model.layers.{i}"] = 0 if i < half else 1
device_map["model.norm"] = 1
device_map["lm_head"]    = 1

# ── Cargar modelo en float16 (sin cuantización) ───────────────────────────────
print("Cargando en float16 (full precision)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map = device_map,
    dtype      = torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

devices = Counter(str(p.device) for p in model.parameters())
print("Distribución:", dict(devices))
assert "cpu" not in devices, "❌ Hay capas en CPU — modelo demasiado grande"
print("✅ Modelo en GPU")
for i in range(n_gpus):
    alloc = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {alloc:.1f}/{total:.1f} GB  (libre: {total-alloc:.1f} GB)")

# ── LoRA (sin gradient checkpointing — no cuantizado) ────────────────────────
# gradient checkpointing no es necesario: pesos en float16 son ligeros
# y tenemos ~8 GB libres por GPU para activaciones
model.enable_input_require_grads()   # requerido para LoRA sin kbit_training

lora_config = LoraConfig(
    r=16, lora_alpha=32,             # rank más alto aprovecha la precisión full
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Dataset ───────────────────────────────────────────────────────────────────
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

train_data = load_jsonl(WORK / 'data/processed/train.jsonl')
valid_data = load_jsonl(WORK / 'data/processed/valid.jsonl')
print(f'Train: {len(train_data)} | Valid: {len(valid_data)}')

def format_sample(s):
    return tokenizer.apply_chat_template(
        s['messages'], tokenize=False, add_generation_prompt=False,
    )

train_ds = Dataset.from_list([{'text': format_sample(s)} for s in train_data])
valid_ds = Dataset.from_list([{'text': format_sample(s)} for s in valid_data])

# ── Entrenamiento ─────────────────────────────────────────────────────────────
NUM_EPOCHS = 1
CKPT_DIR   = str(WORK / 'outputs/checkpoints')

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = train_ds,
    eval_dataset     = valid_ds,
    args = SFTConfig(
        output_dir                  = CKPT_DIR,
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        gradient_checkpointing      = True,   # ahorra activaciones en backward
        warmup_steps                = 50,
        learning_rate               = 1e-4,   # más alto sin cuantización
        lr_scheduler_type           = 'cosine',
        fp16                        = True,
        bf16                        = False,
        logging_steps               = 25,
        eval_strategy               = 'steps',
        eval_steps                  = 200,
        save_strategy               = 'steps',
        save_steps                  = 200,
        save_total_limit            = 3,
        load_best_model_at_end      = True,
        report_to                   = 'none',
        dataloader_pin_memory       = False,
        dataset_text_field          = 'text',
        max_length                  = MAX_SEQ,
    ),
)

checkpoints = sorted(Path(CKPT_DIR).glob('checkpoint-*'), key=os.path.getmtime)
resume_from = str(checkpoints[-1]) if checkpoints else None
if resume_from:
    print(f'Reanudando desde: {resume_from}')

print('Iniciando entrenamiento...')
trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# ── Celda 6: Benchmark con iverilog ──────────────────────────────────────────
import subprocess, tempfile, json
from pathlib import Path
from tqdm.auto import tqdm

model.eval()

def generate(instruction, max_new_tokens=768):
    messages = [
        {'role': 'system', 'content': 'You are an expert RTL designer. Output only synthesisable Verilog.'},
        {'role': 'user',   'content': instruction},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt',
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens,
            temperature=0.1, do_sample=True,
        )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

def compile_check(code):
    with tempfile.NamedTemporaryFile(suffix='.v', mode='w', delete=False) as f:
        f.write(code)
        fname = f.name
    r = subprocess.run(['iverilog', '-o', '/dev/null', '-g2012', fname],
                       capture_output=True, text=True)
    return r.returncode == 0

test_samples = [json.loads(l) for l in open('/kaggle/working/data/processed/test.jsonl')][:30]
passed = 0

for s in tqdm(test_samples, desc='Benchmark'):
    user_msg  = next(m['content'] for m in s['messages'] if m['role'] == 'user')
    generated = generate(user_msg)
    if compile_check(generated):
        passed += 1

rate = passed / len(test_samples) * 100
print(f'\nCompile pass rate: {passed}/{len(test_samples)} ({rate:.1f}%)')

In [ ]:
# ── Celda 7: Exportar a GGUF Q4_K_M ──────────────────────────────────────────
# Primero fusionamos los adaptadores LoRA al modelo base, luego convertimos a GGUF.
# Q4_K_M ≈ 17 GB para 30B — cabe en el output de Kaggle (20 GB)
import subprocess
from pathlib import Path

WORK        = Path('/kaggle/working')
MERGED_DIR  = WORK / 'outputs' / 'merged'
GGUF_DIR    = WORK / 'outputs' / 'gguf'
GGUF_DIR.mkdir(parents=True, exist_ok=True)

# 1. Fusionar LoRA → modelo completo en float16
print('Fusionando adaptadores LoRA...')
merged = model.merge_and_unload()
merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))
print(f'Modelo fusionado guardado en {MERGED_DIR}')

# 2. Instalar llama.cpp para convertir
print('Instalando llama.cpp...')
subprocess.run(['pip', 'install', '-q', 'llama-cpp-python'], check=True)

# 3. Convertir a GGUF con el script oficial de llama.cpp
print('Convirtiendo a GGUF...')
subprocess.run([
    'python', '-m', 'llama_cpp.llama_cpp.convert_hf_to_gguf',
    str(MERGED_DIR),
    '--outfile', str(GGUF_DIR / 'qwen3-coder-verilog-f16.gguf'),
    '--outtype', 'f16',
], check=True)

# 4. Quantizar a Q4_K_M
print('Quantizando a Q4_K_M...')
subprocess.run([
    'python', '-m', 'llama_cpp.llama_cpp.quantize',
    str(GGUF_DIR / 'qwen3-coder-verilog-f16.gguf'),
    str(GGUF_DIR / 'qwen3-coder-verilog-Q4_K_M.gguf'),
    'Q4_K_M',
], check=True)

for f in GGUF_DIR.glob('*.gguf'):
    print(f'{f.name}  →  {f.stat().st_size / 1e9:.2f} GB')

In [ ]:
# ── Celda 8: Subir a HuggingFace Hub ─────────────────────────────────────────
# Opción recomendada — el GGUF es grande para descargar directo desde Kaggle
# Necesitas un token HF con permisos de escritura

from huggingface_hub import HfApi
from pathlib import Path
import glob

HF_TOKEN   = 'hf_TU_TOKEN_AQUI'   # ← reemplaza con tu token
HF_REPO_ID = 'TU_USUARIO/qwen3-coder-verilog'  # ← reemplaza con tu usuario

api = HfApi(token=HF_TOKEN)

# Crear repo si no existe
api.create_repo(repo_id=HF_REPO_ID, repo_type='model', exist_ok=True)

# Subir GGUF
gguf_files = glob.glob('/kaggle/working/outputs/gguf/*.gguf')
for f in gguf_files:
    name = Path(f).name
    print(f'Subiendo {name}...')
    api.upload_file(
        path_or_fileobj = f,
        path_in_repo    = name,
        repo_id         = HF_REPO_ID,
        repo_type       = 'model',
    )
    print(f'  ✅ {name} subido a huggingface.co/{HF_REPO_ID}')

## Instalar en LM Studio

### Opción A — desde HuggingFace Hub
En LM Studio → Search → busca `TU_USUARIO/qwen3-coder-verilog` → Download

### Opción B — descarga manual
```bash
# Descargar con huggingface-cli
huggingface-cli download TU_USUARIO/qwen3-coder-verilog \
    --local-dir ~/Downloads/qwen3-coder-verilog

# Copiar a LM Studio
cp ~/Downloads/qwen3-coder-verilog/*.gguf \
   ~/Library/Application\ Support/LM\ Studio/models/
```

### Tiempos estimados en 2× T4
| Epochs | Tiempo | Notas |
|--------|--------|-------|
| 1 | ~5–7 h | cabe en una sesión Kaggle (9 h) |
| 2 | ~10–14 h | reanudar en segunda sesión |
| 3 | ~15–20 h | reanudar en tercera sesión |